# Laser-detector recall against unseeded human dots (T3)

`p2-tests.md` T3. The production checkpoint (`run3_epoch_021.pt`, sha256 verified in
`run_detector.py`) was run on `sample.csv`: 1,571 frames, 249 dives, stratified by
wavelength × environment, and capped at 12 frames per dive. It ran under two conditions:
`production` (`wavelength=None`, what fishsense-lite sends) and `given` (the human label's
colour, what training and validation supplied).

The detector **always returns a point**, so presence is its confidence at the config's
0.5 threshold. Outcomes are defined in `fishsense_cscw/laser_detection.py`.

In [1]:
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.precision", 3)
from fishsense_cscw import laser_detection as ld
d = ld.load(); prod = d[d.condition == "production"]
ld.summarize(prod)

dot_frames                                              1221
dives                                                    226
recall                                                 0.821
recall_ci           (0.7981236062576058, 0.8411427282627111)
missed                                                 0.115
confident_wrong                                        0.064
no_dot_frames                                            350
false_alarm                                            0.111
false_alarm_ci     (0.08259035674191664, 0.1487040981780249)
dtype: object

In [2]:
prod.groupby(["wavelength", "environment"]).apply(ld.summarize, include_groups=False)

wavelength  environment                 
green       pool         dot_frames                                                 47
                         dives                                                      11
                         recall                                                  0.426
                         recall_ci            (0.2951341600182584, 0.5671833122878847)
                         missed                                                  0.489
                         confident_wrong                                         0.085
            reef         dot_frames                                                400
                         dives                                                      87
                         recall                                                   0.79
                         recall_ci            (0.7474197657946615, 0.8270629026575238)
                         missed                                                  0.185
  

In [3]:
ld.dive_bootstrap(prod)

{'recall_ci': (np.float64(0.7819173455110432), np.float64(0.8566899146107418)),
 'red_minus_green_ci': (np.float64(0.030733671182945278),
  np.float64(0.19300140042217542))}

## Does production's `wavelength=None` cost anything?

In [4]:
d[d.has_dot].groupby(["wavelength", "condition"]).apply(ld.summarize, include_groups=False)[["dot_frames", "recall", "missed", "confident_wrong"]]

dot_frames  recall  missed  confident_wrong
wavelength condition                                              
green      given              447   0.790   0.145            0.065
           production         447   0.752   0.217            0.031
red        given              774   0.873   0.047            0.080
           production         774   0.860   0.057            0.083

In [5]:
pv = d[d.has_dot].pivot_table(index="image_id", columns="condition", values="outcome", aggfunc="first").dropna()
pd.crosstab(pv["production"], pv["given"], rownames=["production"], colnames=["given"])

given,confident-wrong,found,missed
production,,,
confident-wrong,67,6,5
found,5,991,6
missed,19,32,90


## Confidence separates presence well

In [6]:
q = [.05, .25, .5, .75, .95]
pd.DataFrame({k: g.confidence.quantile(q) for k, g in prod.groupby("outcome")}).T

,0.05,0.25,0.50,0.75,0.95
confident-wrong,0.561,0.993,1.000,1.000,1.000
correct-reject,0.005,0.013,0.031,0.073,0.288
false-alarm,0.558,0.733,0.981,1.000,1.000
found,1.000,1.000,1.000,1.000,1.000
missed,0.007,0.024,0.052,0.162,0.381


## Result

**Production recall is 0.821** [0.80, 0.84; 0.78–0.86 resampling whole dives]. 11.5 % of
dots are **missed**, and 6.4 % get a **confident wrong** point, which is a bad measurement
rather than a lost one. On frames with no dot, 11.1 % raise a false alarm.

- **Green is missed about 4× as often as red** (21.7 % vs 5.7 %), for a recall of 0.752
  against 0.860. The gap survives resampling whole dives (+0.03 to +0.19). P1 argues for
  green on detection-reliability grounds, and *this detector* does the opposite. That is
  a claim about the model, not about optics.
- **Pool is harder than reef:** red is 0.940 on reef and 0.736 in the pool, and green
  in the pool is 0.426 (47 frames, 11 dives). This **reverses** `auto_accept.py`'s
  "~93 % on pool, a reef problem". T2 shows the 93 % was labellers accepting seeds.
  Dive 442, the one reef dive behind the "reef problem", was never labelled, so it is
  outside this sample.
- **Withholding the colour costs green ~4 points** (0.752 → 0.790 when given): 32
  green/red frames go from missed to found. Red barely moves. Production's
  `wavelength=None` has a measurable price.

Caveats:

- **Training overlap is unknown**, because the split is not on disk. These numbers sit
  *below* the checkpoint's held-out validation (presence rate red 0.943 vs 0.979, green
  0.783 vs 0.844), which is the opposite of what overlap would do. That is not proof.
- **The human dot is one labeller's**, and the 10 px radius is the auto-accept gate's own
  band.
- **`environment` is a keyword heuristic.**

## Can it be fixed? Three things the same predictions can test

The two failure modes are different animals. **Missed** dots (11.5 %) sit at a median
confidence of 0.05: the detector does not see them. **Confident-wrong** dots (6.4 %) sit at
confidence 1.0, the same as correct ones, and most are a *different bright spot*
entirely.

In [7]:
cw = prod[prod.outcome == "confident-wrong"]
print(f"confident-wrong n={len(cw)}: distance to human dot p10/25/50/75/90 =",
      cw.distance_px.quantile([.1,.25,.5,.75,.9]).round().astype(int).tolist())
print(f"  near miss 10-30 px: {int((cw.distance_px<=30).sum())}   different spot >100 px: {int((cw.distance_px>100).sum())}")

confident-wrong n=78: distance to human dot p10/25/50/75/90 = [12, 47, 592, 838, 1057]
  near miss 10-30 px: 20   different spot >100 px: 57


### Not a fix: retuning the presence threshold

In [8]:
ld.threshold_sweep(prod)

,recall,missed,confident_wrong,false_alarm
threshold,,,,
0.50,0.821,0.115,0.064,0.111
0.30,0.829,0.106,0.066,0.151
0.20,0.834,0.093,0.074,0.189
0.10,0.839,0.076,0.085,0.283
0.05,0.844,0.057,0.099,0.417


Dropping the threshold from 0.5 to 0.1 recovers 1.8 points of recall and raises false
alarms from 11 % to 28 %. The missed dots are not just under the bar.

### Fix 1, free: pass the laser colour (see the production-vs-given table above)

+3.8 points on green, +2.2 overall. Production sends `wavelength=None` and then reads the
colour off the dot *after* predicting (`classify_laser_color`), so a second pass with that
colour is one more inference. Or use the dive's colour: 221 of 262 dives are single-colour.

### Fix 2, cheap: use the dive's laser line at inference, not only post-hoc

The line is the strongest prior in the system and production applies it only after
labelling (auto-accept). Here each prediction is checked against a line through the dive's
*other* human dots, leave-one-out, with at most 11 of them:

In [9]:
ld.line_gate_effect(prod, corridor_px=25.0)

,n,rejected
found,813.0,0.096
confident-wrong,43.0,0.535
false-alarm,19.0,0.842


A 25 px corridor rejects **54 % of the wrong-spot dots and 84 % of false alarms** and flags
9.6 % of correct ones, with lines fitted from a handful of dots. A full dive gives hundreds,
and humans are collinear to about 3 px (P1), so that cost should fall well below 9.6 %.
The inference port already has a line-corridor mode (`DEFAULT_LINE_MASK_CORRIDOR_PX = 25`);
production never passes a line. The line must be **this dive's** (it moves with every
mount change), so the first frames of a dive run unconstrained.

### Fix 3, retrain: needs the training repo and a GPU

Green missed 4× as often as red points to a corpus that is 84 % red. Pool being the worst
environment points to missing hard negatives (white slates, checkerboards, caustics).
There are 46k unseeded human dots to train on, and the checkpoint already carries a
hard-negative sampler. `UCSD-E4E/2026-05-02_laser_detector` is not on this machine.

### Together, without retraining

Fixes 1 + 2: recall ~0.84, wrong-spot measurements roughly halved, false alarms down to
~2 %, residual routed to a human rather than into `Measurement`. The remaining missed
green and pool dots need fix 3. Neither fix 1 nor fix 2 changes the model; both use
information the pipeline already has.